# Scenario 2 — Zero-Shot: No Labelled Training Data
## Encoder Notebook (BERT-base baseline)

**Finding:** BERT with no fine-tuning scores ~50% — the classification head is randomly initialised.
This is the baseline that shows why zero-shot LLMs are needed when no labels exist.

**Dataset:** `puyang2025` — SpamAssassin subset (completely unseen corpus)

**Model:** BERT-base (NO fine-tuning — random head)

In [3]:
!nvidia-smi
!pip install "numpy<2.0" -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall -q
!pip install transformers accelerate bitsandbytes peft datasets scikit-learn pandas tqdm -q

Tue May  5 08:13:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os, re, time, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

ENCODER_MODELS = {
    "BERT":       "bert-base-uncased",
    "RoBERTa":    "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
}

def evaluate(y_true, y_pred, name="", ms=None):
    r = {"Model":name,"Accuracy":f"{accuracy_score(y_true,y_pred):.4f}",
         "Precision":f"{precision_score(y_true,y_pred,zero_division=0):.4f}",
         "Recall":f"{recall_score(y_true,y_pred,zero_division=0):.4f}",
         "F1":f"{f1_score(y_true,y_pred,average='binary',zero_division=0):.4f}"}
    if ms: r["ms/sample"]=f"{ms:.2f}"
    return r

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.enc = tokenizer(list(texts), padding="max_length", truncation=True,
                             max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {k:v[i] for k,v in self.enc.items()}, self.labels[i]


Device: cuda
GPU: Tesla T4


In [6]:
# Dataset: puyang2025/seven-phishing-email-datasets — 203k emails across 7 corpora
print("Loading puyang2025/seven-phishing-email-datasets...")
ds_p = load_dataset("puyang2025/seven-phishing-email-datasets", split="train")
df_puy = ds_p.to_pandas()

# FIX: Changed "body" to "text" as the dataset uses "text" for the email content
df_puy["text"] = (df_puy["subject"].fillna("") + " " + df_puy["text"].fillna("")).str.strip()

df_puy = df_puy[["text","label","dataset_name"]].drop_duplicates("text").dropna().reset_index(drop=True)
df_puy["label"] = df_puy["label"].astype(int)
print(f"Total: {len(df_puy):,}"); print(df_puy["dataset_name"].value_counts().to_string())


Loading puyang2025/seven-phishing-email-datasets...
Total: 162,261
dataset_name
TREC-05     44393
TREC-07     42913
CEAS-08     31144
Enron       23834
TREC-06     13102
Assassin     4584
Ling         2291


In [7]:
# Load SpamAssassin as the held-out test corpus
df_test = df_puy[df_puy["dataset_name"] == "Assassin"].sample(min(500,len(df_puy[df_puy.dataset_name=="Assassin"])), random_state=42)
X_zs, y_zs = df_test["text"].values, df_test["label"].values
print(f"Test set (SpamAssassin): {len(df_test):,} | Legit: {(df_test.label==0).sum()} | Spam: {(df_test.label==1).sum()}")


Test set (SpamAssassin): 500 | Legit: 358 | Spam: 142


In [8]:
# BERT-base with NO fine-tuning — shows what happens without any labelled data
print("Running BERT-base zero-shot (randomly initialised classification head)...")
tok   = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(DEVICE)
model.eval()  # No training at all

preds = []
for i in tqdm(range(0, len(X_zs), 16), desc="BERT zero-shot"):
    enc = tok(list(X_zs[i:i+16]), padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        preds.extend(model(**enc).logits.argmax(-1).cpu().numpy())
del model; torch.cuda.empty_cache()

print("\n" + "="*60)
print("SCENARIO 2 — ENCODER (ZERO-SHOT) RESULTS")
print("="*60)
print(pd.DataFrame([evaluate(y_zs, preds, "BERT-base (zero-shot, no FT)")]).to_string(index=False))
print("\n→ Expected: ~50% accuracy (coin flip — random head)")
print("  This is why LLMs are needed in the zero-shot scenario.")
print("  See Decoder Notebook for LLM zero-shot performance.")


Running BERT-base zero-shot (randomly initialised classification head)...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT zero-shot:   0%|          | 0/32 [00:00<?, ?it/s]


SCENARIO 2 — ENCODER (ZERO-SHOT) RESULTS
                       Model Accuracy Precision Recall     F1
BERT-base (zero-shot, no FT)   0.2860    0.2846 1.0000 0.4431

→ Expected: ~50% accuracy (coin flip — random head)
  This is why LLMs are needed in the zero-shot scenario.
  See Decoder Notebook for LLM zero-shot performance.


In [9]:
# Evaluate all models in the dictionary
all_results = []

for name, model_path in ENCODER_MODELS.items():
    print(f"\nRunning {name} zero-shot (randomly initialised classification head)...")
    tok   = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2).to(DEVICE)
    model.eval()

    preds = []
    start_time = time.time()
    for i in tqdm(range(0, len(X_zs), 16), desc=f"{name} zero-shot"):
        enc = tok(list(X_zs[i:i+16]), padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            preds.extend(model(**enc).logits.argmax(-1).cpu().numpy())
    
    ms_per_sample = (time.time() - start_time) * 1000 / len(X_zs)
    all_results.append(evaluate(y_zs, preds, name, ms_per_sample))
    
    # Clean up memory for the next model
    del model; torch.cuda.empty_cache()

# Display final comparison table
print("\n" + "="*60)
print("SCENARIO 2 — ENCODER (ZERO-SHOT) RESULTS")
print("="*60)
print(pd.DataFrame(all_results).to_string(index=False))



Running BERT zero-shot (randomly initialised classification head)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT zero-shot:   0%|          | 0/32 [00:00<?, ?it/s]


Running RoBERTa zero-shot (randomly initialised classification head)...


...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT zero-shot:   0%|          | 0/32 [00:00<?, ?it/s]


SCENARIO 2 — ENCODER (ZERO-SHOT) RESULTS
     Model Accuracy Precision Recall     F1 ms/sample
      BERT   0.3040    0.2889 0.9930 0.4476     13.57
   RoBERTa   0.7160    0.0000 0.0000 0.0000     13.23
DistilBERT   0.2860    0.2846 1.0000 0.4431      7.61


In [10]:
# 1. Get all unique dataset names
dataset_names = df_puy["dataset_name"].unique()
all_results = []

# 2. Outer Loop: Iterate through each of the 7 datasets
for ds_name in dataset_names:
    print(f"\n{'='*30}\nDATASET: {ds_name}\n{'='*30}")
    
    # Sample 500 emails from the current dataset
    df_curr = df_puy[df_puy["dataset_name"] == ds_name].sample(
        min(500, len(df_puy[df_puy.dataset_name == ds_name])), 
        random_state=42
    )
    X_zs, y_zs = df_curr["text"].values, df_curr["label"].values
    print(f"Test set: {len(df_curr):,} | Legit: {(df_curr.label==0).sum()} | Spam: {(df_curr.label==1).sum()}")

    # 3. Inner Loop: Iterate through each of the 3 models
    for model_name, model_path in ENCODER_MODELS.items():
        print(f"Running {model_name}...")
        
        tok = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2).to(DEVICE)
        model.eval()
        
        preds = []
        for i in tqdm(range(0, len(X_zs), 16), desc=f"{model_name} on {ds_name}", leave=False):
            enc = tok(list(X_zs[i:i+16]), padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                preds.extend(model(**enc).logits.argmax(-1).cpu().numpy())
        
        # Evaluate and store results
        res = evaluate(y_zs, preds, model_name)
        res["Dataset"] = ds_name
        all_results.append(res)
        
        # Clean up memory
        del model, tok
        if torch.cuda.is_available(): torch.cuda.empty_cache()

# 4. Final Summary Table
print("\n" + "="*60)
print("SCENARIO 2 — ALL MODELS & ALL DATASETS RESULTS")
print("="*60)
df_results = pd.DataFrame(all_results)
# Reorder columns for better readability
cols = ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1"]
print(df_results[cols].to_string(index=False))



DATASET: TREC-07
Test set: 500 | Legit: 216 | Spam: 284
Running BERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT on TREC-07:   0%|          | 0/32 [00:00<?, ?it/s]

Running RoBERTa...


...

Running DistilBERT...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT on Ling:   0%|          | 0/32 [00:00<?, ?it/s]


SCENARIO 2 — ALL MODELS & ALL DATASETS RESULTS
 Dataset      Model Accuracy Precision Recall     F1
 TREC-07       BERT   0.5180    0.7363 0.2359 0.3573
 TREC-07    RoBERTa   0.4340    0.5556 0.0176 0.0341
 TREC-07 DistilBERT   0.4360    1.0000 0.0070 0.0140
 TREC-05       BERT   0.4460    0.2318 0.1786 0.2017
 TREC-05    RoBERTa   0.3920    0.3920 1.0000 0.5632
 TREC-05 DistilBERT   0.6080    0.0000 0.0000 0.0000
 TREC-06       BERT   0.7760    0.6667 0.0690 0.1250
 TREC-06    RoBERTa   0.7680    0.0000 0.0000 0.0000
 TREC-06 DistilBERT   0.2760    0.2085 0.7586 0.3271
   Enron       BERT   0.4020    0.3806 0.3917 0.3860
   Enron    RoBERTa   0.5200    0.0000 0.0000 0.0000
   Enron DistilBERT   0.4800    0.4800 1.0000 0.6486
Assassin       BERT   0.6860    0.4444 0.4225 0.4332
Assassin    RoBERTa   0.2840    0.2840 1.0000 0.4424
Assassin DistilBERT   0.2860    0.2846 1.0000 0.4431
 CEAS-08       BERT   0.6120    0.6547 0.6500 0.6523
 CEAS-08    RoBERTa   0.5600    0.5600 1.0000 0.717